<a href="https://colab.research.google.com/github/EvitaKyprianou/Genomics-portfolio/blob/main/variant_lookup_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# SNP Lookup Agent -- Ensembl REST API
# For: MSc Genomic Medicine, Google Colab
# ============================================================

import requests
import time
import json

snp_ids = ["rs429358", "rs7412", "rs1801133", "rs334"]

base_url = "https://rest.ensembl.org"

headers = {
    "Content-Type": "application/json"
}

results = []

for rsid in snp_ids:

    print(f"Looking up {rsid}...")

    endpoint = f"/variation/human/{rsid}?content-type=application/json"
    url = base_url + endpoint

    response = requests.get(url, headers=headers)

    if response.status_code == 200:

        data = response.json()

        mappings = data.get("mappings", [])

        if mappings:
            first_mapping = mappings[0]
            chromosome = first_mapping.get("seq_region_name", "N/A")
            position   = first_mapping.get("start", "N/A")
            allele_string = first_mapping.get("allele_string", "N/A")
        else:
            chromosome    = "N/A"
            position      = "N/A"
            allele_string = "N/A"

        gene_name = data.get("source", "see Ensembl")

        results.append({
            "rsID":        rsid,
            "Chromosome":  chromosome,
            "Position":    position,
            "Source":      gene_name,
            "Consequence": data.get("most_severe_consequence", "N/A"),
            "Alleles":     allele_string
        })

    else:
        print(f"  Warning: Error {response.status_code} for {rsid}")
        results.append({
            "rsID":        rsid,
            "Chromosome":  "ERROR",
            "Position":    "ERROR",
            "Source":      "ERROR",
            "Consequence": "ERROR",
            "Alleles":     "ERROR"
        })

    time.sleep(0.5)

print("\n" + "="*65)
print("  SNP LOOKUP SUMMARY -- Ensembl REST API (GRCh38)")
print("="*65)
print(f"{'rsID':<12} {'Chr':<5} {'Position':<12} {'Consequence':<28}")
print("-"*65)

for r in results:
    print(
        f"{r['rsID']:<12} "
        f"{r['Chromosome']:<5} "
        f"{r['Position']:<12} "
        f"{r['Consequence']:<28}"
    )

print("="*65)
print(f"Total SNPs queried: {len(results)}")
print(f"Reference genome:   GRCh38 (hg38)")
print(f"API endpoint:       rest.ensembl.org/variation/human/")
print("="*65)

Looking up rs429358...
Looking up rs7412...
Looking up rs1801133...
Looking up rs334...

  SNP LOOKUP SUMMARY -- Ensembl REST API (GRCh38)
rsID         Chr   Position     Consequence                 
-----------------------------------------------------------------
rs429358     19    44908684     missense_variant            
rs7412       19    44908822     missense_variant            
rs1801133    1     11796321     missense_variant            
rs334        11    5227002      missense_variant            
Total SNPs queried: 4
Reference genome:   GRCh38 (hg38)
API endpoint:       rest.ensembl.org/variation/human/


In [9]:
# ============================================================
# SNP Lookup Agent v2 -- Ensembl + gnomAD
# Adds allele frequency lookup and rare variant flagging
# ============================================================

import requests
import time
import json

snp_ids = ["rs429358", "rs7412", "rs1801133", "rs334"]

# --- Ensembl setup (unchanged from v1) ---

ensembl_url = "https://rest.ensembl.org"
ensembl_headers = {"Content-Type": "application/json"}

# --- gnomAD setup (new) ---

gnomad_url = "https://gnomad.broadinstitute.org/api"

# gnomAD uses GraphQL -- a query language where you describe
# exactly what fields you want back. The server is at one URL
# and you POST a query string to it each time.
# This is different from Ensembl where each data type has
# its own endpoint URL.

gnomad_headers = {"Content-Type": "application/json"}

# --- Threshold for "rare" classification ---

RARE_THRESHOLD = 0.01

# 0.01 = 1% minor allele frequency.
# This is the standard population genetics cutoff used in
# ACMG variant classification guidelines (Richards et al., 2015,
# Genetics in Medicine) to distinguish common polymorphisms
# from potentially clinically relevant rare variants.
# gnomAD returns AF as a decimal: 0.01 = 1%, 0.001 = 0.1% etc.

# --- Function 1: Query Ensembl for a single SNP ---

def get_ensembl_data(rsid):
    """
    Sends a GET request to the Ensembl variation endpoint.
    Returns a dictionary with chromosome, position, consequence.
    Returns None if the request fails.
    """
    endpoint = f"/variation/human/{rsid}?content-type=application/json"
    url = ensembl_url + endpoint

    response = requests.get(url, headers=ensembl_headers)

    if response.status_code == 200:
        data = response.json()
        mappings = data.get("mappings", [])

        if mappings:
            first = mappings[0]
            return {
                "chromosome":   first.get("seq_region_name", "N/A"),
                "position":     first.get("start", "N/A"),
                "consequence":  data.get("most_severe_consequence", "N/A"),
                "alleles":      first.get("allele_string", "N/A")
            }
    return None

# def defines a reusable function -- a named block of code
# you can call multiple times with different inputs.
# 'rsid' inside the brackets is the parameter: a placeholder
# that gets replaced with the actual rsID when you call it.
# 'return' sends data back to wherever the function was called.
# Returning None signals that something went wrong.

# --- Function 2: Query gnomAD for allele frequency ---

def get_gnomad_af(rsid, chromosome, position, alleles):
    """
    Queries the gnomAD GraphQL API for the global allele
    frequency of a variant.

    gnomAD identifies variants by genomic coordinates, not rsID.
    Format required: chromosome-position-ref-alt
    e.g. 19-44908684-T-C

    Returns a float (the AF) or None if not found.
    """

    # Parse the ref and alt alleles from the Ensembl allele string
    # Ensembl returns this as e.g. "T/C" meaning REF=T, ALT=C

    if "/" not in str(alleles):
        return None

    parts = alleles.split("/")
    ref = parts[0].strip()
    alt = parts[1].strip()

    # .split("/") splits the string "T/C" into a list ["T", "C"]
    # parts[0] is the reference allele, parts[1] is the alternate
    # .strip() removes any accidental whitespace

    variant_id = f"{chromosome}-{position}-{ref}-{alt}"

    # This builds gnomAD's required variant identifier string.
    # e.g. "19-44908684-T-C"

    # --- Build the GraphQL query ---

    query = """
    {
      variant(variantId: "%s", dataset: gnomad_r4) {
        genome {
          af
        }
      }
    }
    """ % variant_id

    # This is a GraphQL query string.
    # It says: "from the gnomAD r4 (v4) dataset, find the variant
    # with this ID, and return its genome allele frequency (af)."
    # gnomad_r4 is the most current major release (2023), covering
    # ~730,000 exomes and ~76,000 genomes from diverse populations.
    # %s is a string formatting placeholder -- % variant_id
    # inserts the actual variant ID into the query.

    # --- Send the POST request to gnomAD ---

    payload = {"query": query}

    # GraphQL always uses POST requests (not GET).
    # The query is sent as JSON in the request body,
    # wrapped in a dictionary with the key "query".

    try:
        response = requests.post(
            gnomad_url,
            headers=gnomad_headers,
            json=payload,
            timeout=15
        )

        # requests.post() sends a POST request.
        # json=payload automatically converts the dictionary
        # to a JSON string and sends it in the request body.
        # timeout=15 means: if no reply within 15 seconds,
        # give up and raise an error instead of hanging forever.

        if response.status_code == 200:
            data = response.json()

            # Navigate the nested JSON response:
            # data -> "data" -> "variant" -> "genome" -> "af"

            variant_data = data.get("data", {}).get("variant")

            if variant_data is None:
                return None
            # gnomAD returns null for variants not in its database

            genome_data = variant_data.get("genome")
            if genome_data is None:
                return None

            af = genome_data.get("af")
            return af
            # af will be a float like 0.2356 (23.56%)
            # or None if the field is missing

    except requests.exceptions.Timeout:
        print(f"  gnomAD timed out for {variant_id}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  gnomAD request error: {e}")
        return None

    # try/except catches errors so the whole script doesn't
    # crash if one gnomAD lookup fails.
    # RequestException is a broad catch for any network problem.

# --- Function 3: Classify the allele frequency ---

def classify_af(af):
    """
    Takes an allele frequency float and returns a classification
    string and a flag string.
    """
    if af is None:
        return "N/A", "not in gnomAD"
    elif af < RARE_THRESHOLD:
        return f"{af:.4f}", "*** RARE ***"
    else:
        return f"{af:.4f}", "common"

    # f"{af:.4f}" formats the float to 4 decimal places.
    # e.g. 0.00234 becomes "0.0023"
    # This keeps the table readable.
    # "*** RARE ***" is the flag shown for variants below 1%.

# --- Main loop ---

results = []

for rsid in snp_ids:
    print(f"Processing {rsid}...")

    # Step 1: Ensembl lookup
    ensembl = get_ensembl_data(rsid)
    time.sleep(0.5)

    if ensembl is None:
        print(f"  Ensembl lookup failed for {rsid}")
        results.append({
            "rsID": rsid,
            "Chr": "ERROR", "Position": "ERROR",
            "Consequence": "ERROR",
            "AF": "ERROR", "Flag": "ERROR"
        })
        continue
    # 'continue' skips the rest of this loop iteration
    # and moves on to the next rsID in the list.

    # Step 2: gnomAD lookup using coordinates from Ensembl
    af_raw = get_gnomad_af(
        rsid,
        ensembl["chromosome"],
        ensembl["position"],
        ensembl["alleles"]
    )
    time.sleep(0.5)

    af_str, flag = classify_af(af_raw)

    # classify_af returns two values at once.
    # Python lets you unpack them directly:
    # af_str, flag = ("0.0023", "*** RARE ***")

    results.append({
        "rsID":        rsid,
        "Chr":         ensembl["chromosome"],
        "Position":    ensembl["position"],
        "Consequence": ensembl["consequence"],
        "AF":          af_str,
        "Flag":        flag
    })

    print(f"  Chr {ensembl['chromosome']}  AF={af_str}  {flag}")

# --- Print final summary table ---

print("\n" + "="*72)
print("  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)")
print("="*72)
print(f"{'rsID':<12} {'Chr':<4} {'Position':<12} {'AF (global)':<14} {'Flag'}")
print("-"*72)

for r in results:
    print(
        f"{r['rsID']:<12} "
        f"{r['Chr']:<4} "
        f"{r['Position']:<12} "
        f"{r['AF']:<14} "
        f"{r['Flag']}"
    )

print("="*72)
print(f"Rare threshold:   AF < {RARE_THRESHOLD} ({RARE_THRESHOLD*100}%)")
print(f"gnomAD dataset:   gnomad_r4 (v4, GRCh38)")
print(f"AF type:          global genome allele frequency")
print("="*72)

Processing rs429358...
  Chr 19  AF=0.1574  common
Processing rs7412...
  Chr 19  AF=0.0778  common
Processing rs1801133...
  Chr 1  AF=0.2752  common
Processing rs334...
  Chr 11  AF=0.0127  common

  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)
rsID         Chr  Position     AF (global)    Flag
------------------------------------------------------------------------
rs429358     19   44908684     0.1574         common
rs7412       19   44908822     0.0778         common
rs1801133    1    11796321     0.2752         common
rs334        11   5227002      0.0127         common
Rare threshold:   AF < 0.01 (1.0%)
gnomAD dataset:   gnomad_r4 (v4, GRCh38)
AF type:          global genome allele frequency


In [10]:
# ============================================================
# SNP Lookup Agent v3 -- Ensembl + gnomAD + CSV export
# ============================================================

import requests
import time
import json
import csv        # NEW -- built into Python, no installation needed
import os         # NEW -- lets us find and print the file path

with open("snp_input.csv", "r") as f:
    reader = csv.reader(f)
    snp_ids = [row[0].strip() for row in reader if row]

print(f"Loaded {len(snp_ids)} variants from file")

ensembl_url = "https://rest.ensembl.org"
ensembl_headers = {"Content-Type": "application/json"}

gnomad_url = "https://gnomad.broadinstitute.org/api"
gnomad_headers = {"Content-Type": "application/json"}

RARE_THRESHOLD = 0.01

# ---- Function 1: Ensembl lookup (unchanged) ----------------

def get_ensembl_data(rsid):
    endpoint = f"/variation/human/{rsid}?content-type=application/json"
    url = ensembl_url + endpoint
    response = requests.get(url, headers=ensembl_headers)
    if response.status_code == 200:
        data = response.json()
        mappings = data.get("mappings", [])
        if mappings:
            first = mappings[0]
            return {
                "chromosome":  first.get("seq_region_name", "N/A"),
                "position":    first.get("start", "N/A"),
                "consequence": data.get("most_severe_consequence", "N/A"),
                "alleles":     first.get("allele_string", "N/A")
            }
    return None

# ---- Function 2: gnomAD lookup (unchanged) -----------------

def get_gnomad_af(rsid, chromosome, position, alleles):
    if "/" not in str(alleles):
        return None
    parts = alleles.split("/")
    ref = parts[0].strip()
    alt = parts[1].strip()
    variant_id = f"{chromosome}-{position}-{ref}-{alt}"
    query = """
    {
      variant(variantId: "%s", dataset: gnomad_r4) {
        genome {
          af
        }
      }
    }
    """ % variant_id
    payload = {"query": query}
    try:
        response = requests.post(
            gnomad_url,
            headers=gnomad_headers,
            json=payload,
            timeout=15
        )
        if response.status_code == 200:
            data = response.json()
            variant_data = data.get("data", {}).get("variant")
            if variant_data is None:
                return None
            genome_data = variant_data.get("genome")
            if genome_data is None:
                return None
            return genome_data.get("af")
    except requests.exceptions.Timeout:
        print(f"  gnomAD timed out for {variant_id}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  gnomAD request error: {e}")
        return None

# ---- Function 3: AF classification (unchanged) -------------

def classify_af(af):
    if af is None:
        return "N/A", "not in gnomAD"
    elif af < RARE_THRESHOLD:
        return f"{af:.6f}", "RARE"
    else:
        return f"{af:.6f}", "common"

# af:.6f gives 6 decimal places in the CSV -- more precise
# than the 4 used for screen printing, which matters when
# AF values are very small (e.g. 0.000023)

# ---- Function 4: CSV export (NEW) -------------------------

def export_to_csv(results, filename="variant_report.csv"):
    """
    Writes the results list to a CSV file.

    csv.DictWriter takes a list of dictionaries and writes
    each one as a row, using the dictionary keys as column
    headers automatically.
    """

    # Define the column order explicitly.
    # DictWriter uses this list to decide which keys to write
    # and in what order -- left to right in the CSV.
    fieldnames = [
        "rsID",
        "Chromosome",
        "Position",
        "Alleles",
        "Consequence",
        "AF_global",
        "Frequency_class",
        "gnomAD_dataset",
        "Reference_genome"
    ]

    with open(filename, "w", newline="", encoding="utf-8") as csvfile:

        # open() creates or overwrites the file.
        # "w" means write mode -- it starts fresh each run.
        # newline="" is important on Windows: without it,
        # Python adds an extra blank line between every row.
        # encoding="utf-8" ensures special characters are safe.

        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # csv.DictWriter is a writer that accepts dictionaries.
        # Each dictionary key maps to a column name.
        # fieldnames tells it which keys to expect and in
        # what order to write them.

        writer.writeheader()

        # writeheader() writes the first row:
        # rsID,Chromosome,Position,Alleles,Consequence,...
        # This is what Excel will use as column names.

        for r in results:
            writer.writerow(r)

        # writerow() writes one dictionary as one CSV row.
        # The loop runs once per SNP in results.

    # Confirm where the file was saved
    abs_path = os.path.abspath(filename)
    print(f"\nCSV saved: {abs_path}")
    print(f"Rows written: {len(results)}")

    # os.path.abspath() converts the filename to a full path
    # e.g. /root/ or /content/variant_report.csv in Colab.
    # This tells you exactly where to find the file.

# ---- Main loop ---------------------------------------------

results = []

for rsid in snp_ids:
    print(f"Processing {rsid}...")

    ensembl = get_ensembl_data(rsid)
    time.sleep(0.5)

    if ensembl is None:
        print(f"  Ensembl lookup failed for {rsid}")
        results.append({
            "rsID":              rsid,
            "Chromosome":        "ERROR",
            "Position":          "ERROR",
            "Alleles":           "ERROR",
            "Consequence":       "ERROR",
            "AF_global":         "ERROR",
            "Frequency_class":   "ERROR",
            "gnomAD_dataset":    "gnomad_r4",
            "Reference_genome":  "GRCh38"
        })
        continue

    af_raw = get_gnomad_af(
        rsid,
        ensembl["chromosome"],
        ensembl["position"],
        ensembl["alleles"]
    )
    time.sleep(0.5)

    af_str, flag = classify_af(af_raw)

    results.append({
        "rsID":              rsid,
        "Chromosome":        ensembl["chromosome"],
        "Position":          ensembl["position"],
        "Alleles":           ensembl["alleles"],
        "Consequence":       ensembl["consequence"],
        "AF_global":         af_str,
        "Frequency_class":   flag,
        "gnomAD_dataset":    "gnomad_r4",
        "Reference_genome":  "GRCh38"
    })

    print(f"  Chr {ensembl['chromosome']}  AF={af_str}  {flag}")

# ---- Print summary to screen (kept from v2) ----------------

print("\n" + "="*72)
print("  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)")
print("="*72)
print(f"{'rsID':<12} {'Chr':<4} {'Position':<12} {'AF (global)':<14} {'Flag'}")
print("-"*72)

for r in results:
    print(
        f"{r['rsID']:<12} "
        f"{r['Chromosome']:<4} "
        f"{r['Position']:<12} "
        f"{r['AF_global']:<14} "
        f"{r['Frequency_class']}"
    )

print("="*72)

# ---- Export to CSV (NEW -- one line to call the function) --

export_to_csv(results, filename="variant_report.csv")

Loaded 4 variants from file
Processing ﻿rs429358...
  Ensembl lookup failed for ﻿rs429358
Processing rs7412...
  Chr 19  AF=0.077837  common
Processing rs1801133...
  Chr 1  AF=0.275170  common
Processing rs334...
  Chr 11  AF=0.012719  common

  SNP SUMMARY -- Ensembl + gnomAD v4 (GRCh38)
rsID         Chr  Position     AF (global)    Flag
------------------------------------------------------------------------
﻿rs429358    ERROR ERROR        ERROR          ERROR
rs7412       19   44908822     0.077837       common
rs1801133    1    11796321     0.275170       common
rs334        11   5227002      0.012719       common

CSV saved: /content/variant_report.csv
Rows written: 4
